# QAOA for RF Security in PennyLane

Two applications with synthetic data:
1. PSD-based anomaly scoring (benign vs fake-SSB vs tone) using contrastive Ising costs.
2. RACH preamble detection with reference-normalized correlation features and contrastive score.

QAOA depth kept small for speed; objectives use mini-batch training; plots use default Matplotlib settings.

In [ ]:
import pennylane as qml
from pennylane import numpy as np
import numpy as base_np
import numpy.fft as fft
import matplotlib.pyplot as plt
print('PennyLane:', qml.__version__)

## Synthetic RF helpers (PSD and correlation features)

In [ ]:
def gen_ofdm_like(N=1024, subcarriers=(60,120,200), amps=(1.0,0.8,0.6), noise=0.05, fs=1.0):
    t = base_np.arange(N)/fs
    x = base_np.zeros(N, dtype=complex)
    for f,a in zip(subcarriers,amps):
        x += a * base_np.exp(1j*2*base_np.pi*f*t/N)
    x += noise * (base_np.random.randn(N) + 1j*base_np.random.randn(N))
    return x
def add_tone_jammer(x, tone_bin=150, amp=3.0):
    N=len(x); n=base_np.arange(N)
    return x + amp*base_np.exp(1j*2*base_np.pi*tone_bin*n/N)
def shift_sync(x, shift_bins=6):
    X=fft.fft(x); X=base_np.roll(X, shift_bins); return fft.ifft(X)
def psd_feature(x, n_bins=16):
    P = base_np.abs(fft.fft(x))**2
    k = len(P)//n_bins
    P = P[:k*n_bins].reshape(n_bins, k).mean(axis=1)
    P = P/(P.max()+1e-12)
    return P.astype(float)
def zadoff_chu(u,Nzc):
    n = base_np.arange(Nzc)
    return base_np.exp(-1j*base_np.pi*u*n*(n+1)/Nzc)
def rach_burst(zc, delay=0, snr_db=12.0):
    N=len(zc); x=base_np.zeros(N, dtype=complex); x[delay:]=zc[:N-delay]
    snr_lin = 10**(snr_db/10)
    sig_power = base_np.mean(base_np.abs(x)**2)+1e-12
    noise_power = sig_power/snr_lin
    noise = base_np.sqrt(noise_power/2)*(base_np.random.randn(N)+1j*base_np.random.randn(N))
    return x + noise
def corr_feature_refnorm(rx, ref, n_bins=16, ref_scale=None):
    r = base_np.correlate(rx, ref, mode='same')
    c = base_np.abs(r)
    if ref_scale is None:
        rref = base_np.correlate(ref, ref, mode='same')
        ref_scale = float(base_np.max(base_np.abs(rref)))
    c = c/(ref_scale+1e-12)
    k = len(c)//n_bins
    c = c[:k*n_bins].reshape(n_bins, k).mean(axis=1)
    return c.astype(float)

## QAOA building blocks and Ising Hamiltonians

In [ ]:
def make_device(B):
    return qml.device('default.qubit', wires=B)
def angle_embed(x, wires):
    qml.AngleEmbedding(x, wires=wires, rotation='Y')
def qaoa_block(gammas, betas, H, wires):
    for g, b in zip(gammas, betas):
        qml.qaoa.cost_layer(g, H)
        mixer_h = qml.qaoa.x_mixer(wires)
    qml.qaoa.mixer_layer(b, mixer_h)
def build_ising_H_from_stats(mean_vec, cov_mat, wires, kappa=1.0):
    H = 0
    mu = mean_vec - mean_vec.mean()
    for wi, a in zip(wires, mu):
        H = H + float(a)*qml.PauliZ(wi)
    for i in range(len(wires)-1):
        J = kappa*float(cov_mat[i, i+1])
        if abs(J) > 1e-12:
            H = H + J*(qml.PauliZ(wires[i]) @ qml.PauliZ(wires[i+1]))
    return H
def scalar(x):
    try:
        return float(x)
    except Exception:
        return float(np.array(x))

## Task 1: PSD anomaly scoring with contrastive QAOA

In [ ]:
base_np.random.seed(3)
N = 1024; B = 16; p = 2
wires = list(range(B))
dev_psd = make_device(B)

@qml.qnode(dev_psd)
def energy_qaoa_psd(params, x, Hcost):
    gammas = params[:p]; betas = params[p:2*p]
    angle_embed(x, wires)
    qaoa_block(gammas, betas, Hcost, wires)
    return qml.expval(Hcost)

def batch_energy(params, X, H):
    vals = [energy_qaoa_psd(params, x, H) for x in X]
    return np.mean(np.stack(vals))

def loss_psd(params, Xb, Xf, Hb, Hf, margin=0.15, lam=1.0):
    Eb = batch_energy(params, Xb, Hb)
    Ef = batch_energy(params, Xf, Hf)
    gap = Ef - Eb
    return Eb + lam * qml.math.maximum(0.0, margin - gap)

# Data: benign vs fake (+ optional tone)
def sample_psd_set(kind, n=64):
    S = []
    for _ in range(n):
        x = gen_ofdm_like(N=N)
        if kind == 'fake':
            x = shift_sync(x, shift_bins=6)
        elif kind == 'tone':
            x = add_tone_jammer(x, tone_bin=150, amp=3.0)
        S.append(psd_feature(x, n_bins=B))
    return base_np.stack(S, axis=0)

S_b = sample_psd_set('benign', n=64)
S_f = sample_psd_set('fake',   n=64)
mu_b, mu_f = S_b.mean(axis=0), S_f.mean(axis=0)
Cb, Cf = base_np.cov(S_b.T), base_np.cov(S_f.T)
Hb = build_ising_H_from_stats(mu_b, Cb, wires, kappa=1.0)
Hf = build_ising_H_from_stats(mu_f, Cf, wires, kappa=1.0)

np.random.seed(0)
params_psd = np.random.uniform(low=-0.1, high=0.1, size=2*p)
opt = qml.AdamOptimizer(stepsize=0.05)
loss_trace = []
for step in range(60):
    idx = (step*8) % 56
    Xb = S_b[idx:idx+8]
    Xf = S_f[idx:idx+8]
    params_psd, curr = opt.step_and_cost(lambda th: loss_psd(th, Xb, Xf, Hb, Hf, margin=0.15, lam=1.0), params_psd)
    loss_trace.append(scalar(curr))
print('Final PSD loss:', round(loss_trace[-1], 4))

plt.figure(figsize=(6,4))
plt.plot(loss_trace)
plt.xlabel('Step'); plt.ylabel('Loss'); plt.title('PSD QAOA training loss')
plt.show()

def deltaE_psd(params, x):
    Ef = energy_qaoa_psd(params, x, Hf)
    Eb = energy_qaoa_psd(params, x, Hb)
    return Ef - Eb

dE_b = np.stack([deltaE_psd(params_psd, x) for x in S_b[56:]])
dE_f = np.stack([deltaE_psd(params_psd, x) for x in S_f[56:]])
print({'ΔE mean (fake - benign)': scalar(np.mean(dE_f) - np.mean(dE_b))})

plt.figure(figsize=(6,4))
plt.hist([np.array(dE_b), np.array(dE_f)], bins=15, alpha=0.7, label=['benign','fake'])
plt.xlabel('ΔE = <H_fake> - <H_benign>'); plt.ylabel('Count'); plt.title('PSD validation ΔE')
plt.legend(); plt.show()

## Task 2: RACH preamble detection with contrastive QAOA (ref-normalized)

In [ ]:
base_np.random.seed(11)
Nzc = 839; u_root = 167
zc = zadoff_chu(u_root, Nzc)
Bcorr = 16; pR = 2
wiresR = list(range(Bcorr))
dev_rach = make_device(Bcorr)

@qml.qnode(dev_rach)
def energy_qaoa_rach(params, x, Hcost):
    gammas = params[:pR]; betas = params[pR:2*pR]
    angle_embed(x, wiresR)
    qaoa_block(gammas, betas, Hcost, wiresR)
    return qml.expval(Hcost)

def batch_energy_R(params, X, H):
    vals = [energy_qaoa_rach(params, x, H) for x in X]
    return np.mean(np.stack(vals))

rref = base_np.correlate(zc, zc, mode='same')
ref_scale = float(base_np.max(base_np.abs(rref)))
R_pos = base_np.stack([corr_feature_refnorm(rach_burst(zc, delay=12+d, snr_db=12.0), zc, n_bins=Bcorr, ref_scale=ref_scale) for d in range(-3,4)])
R_neg = base_np.stack([corr_feature_refnorm(rach_burst(zc, delay=base_np.random.randint(40,120), snr_db=8.0), zc, n_bins=Bcorr, ref_scale=ref_scale) for _ in range(28)])

mu_p, mu_n = R_pos.mean(axis=0), R_neg.mean(axis=0)
Cp, Cn = base_np.cov(R_pos.T), base_np.cov(R_neg.T)
Hpos = build_ising_H_from_stats(mu_p, Cp, wiresR, kappa=1.0)
Hneg = build_ising_H_from_stats(mu_n, Cn, wiresR, kappa=1.0)

def loss_rach(params, Rp, Rn, margin=0.08, lam=1.0):
    Ep = batch_energy_R(params, Rp, Hpos)
    En = batch_energy_R(params, Rn, Hneg)
    gap = En - Ep
    return Ep + lam * qml.math.maximum(0.0, margin - gap)

np.random.seed(2)
params_rach = np.random.uniform(low=-0.1, high=0.1, size=2*pR)
optR = qml.AdamOptimizer(stepsize=0.05)
loss_trace_R = []
for step in range(60):
    idx = (step*6) % 21
    Rp = R_pos[max(0,idx-3):idx+4]
    Rn = R_neg[idx:idx+7]
    params_rach, curr = optR.step_and_cost(lambda th: loss_rach(th, Rp, Rn, margin=0.08, lam=1.0), params_rach)
    loss_trace_R.append(scalar(curr))
print('Final RACH loss:', round(loss_trace_R[-1], 4))

plt.figure(figsize=(6,4))
plt.plot(loss_trace_R)
plt.xlabel('Step'); plt.ylabel('Loss'); plt.title('RACH QAOA training loss')
plt.show()

def deltaE_rach(params, x):
    En = energy_qaoa_rach(params, x, Hneg)
    Ep = energy_qaoa_rach(params, x, Hpos)
    return En - Ep

x_match = corr_feature_refnorm(rach_burst(zc, delay=12, snr_db=12.0), zc, n_bins=Bcorr, ref_scale=ref_scale)
x_mism  = corr_feature_refnorm(rach_burst(zc, delay=70, snr_db=8.0),  zc, n_bins=Bcorr, ref_scale=ref_scale)
print({'ΔE_RACH (matched)': scalar(deltaE_rach(params_rach, x_match)),
       'ΔE_RACH (mismatched)': scalar(deltaE_rach(params_rach, x_mism))})

### Notes
- Increase bins (B, Bcorr) for finer resolution; increase QAOA depth p for capacity (training cost grows).
- Use SPSA for shot-noisy hardware; try local mixers if needed.
- Extend to multiple RACH hypotheses by building a small head bank $\{H_{\text{pos}}^{(c)},H_{\text{neg}}^{(c)}\}$ and selecting the minimum-energy head.